In [ ]:
#==================================
# 地形指标提取脚本 - 获取 Mean_Slope Northness Eastness Solar_Radiation_Index Terrain_Heterogeneity
# 作者: King
# 日期: 2026-02-19
#==================================

import arcpy
import os
import math
import csv
import time

# ==================== 参数配置 ====================
# 1. 包含你之前下载好的 1822 个小 DEM 文件的文件夹
dem_folder = r"C:\Users\King\Desktop\大三下学习生活\prj-Alban\data\raw\DEM\DEM_Evelation"
# 2. 结果导出路径
output_csv = r"C:\Users\King\Desktop\大三下学习生活\prj-Alban\data\output\Bird_Habitat_Terrain_Stats.csv"
# 3. 你的缓冲区图层名称
buffer_layer = "Buffer"

# 检查路径
if not os.path.exists(os.path.dirname(output_csv)):
    os.makedirs(os.path.dirname(output_csv))

# 启用空间分析扩展
arcpy.CheckOutExtension("Spatial")

# ==================== 初始化 Debug 与监控变量 ====================
res = arcpy.management.GetCount(buffer_layer)
total_points = int(res[0])
results = []
header = ["OID", "Mean_Slope", "Northness", "Eastness", "Solar_Radiation_Index", "Terrain_Heterogeneity"]

start_time = time.time()
processed_count = 0
error_count = 0

print(f"--- 任务启动：共计 {total_points} 个样点准备分析 ---")
arcpy.AddMessage(f"开始地形指标提取，总数: {total_points}")

# ==================== 执行核心逻辑 ====================
with arcpy.da.SearchCursor(buffer_layer, ["OID@"]) as cursor:
    for row in cursor:
        oid = row[0]
        dem_path = os.path.join(dem_folder, f"Extract_DEM_{oid}.tif")
        
        # Debug 检查文件是否存在
        if not os.path.exists(dem_path):
            # print(f"跳过: 找不到文件 {dem_path}") # 如果需要非常详细的debug可以取消注释
            continue
        
        try:
            # 基础地形计算
            slope_rast = arcpy.sa.Slope(dem_path, "DEGREE")
            aspect_rast = arcpy.sa.Aspect(dem_path)
            
            # 转换弧度并计算 Northness/Eastness
            aspect_rad = arcpy.sa.Raster(aspect_rast) * (math.pi / 180.0)
            northness_rast = arcpy.sa.Cos(aspect_rad)
            eastness_rast = arcpy.sa.Sin(aspect_rad)
            
            # 计算潜在太阳辐射指数 (PSR)
            slope_rad = arcpy.sa.Raster(slope_rast) * (math.pi / 180.0)
            psr_rast = arcpy.sa.Cos(slope_rad) + (arcpy.sa.Sin(slope_rad) * arcpy.sa.Cos(aspect_rad))
            
            # 提取统计值
            m_slope = float(arcpy.management.GetRasterProperties(slope_rast, "MEAN").getOutput(0))
            std_slope = float(arcpy.management.GetRasterProperties(slope_rast, "STD").getOutput(0))
            m_north = float(arcpy.management.GetRasterProperties(northness_rast, "MEAN").getOutput(0))
            m_east = float(arcpy.management.GetRasterProperties(eastness_rast, "MEAN").getOutput(0))
            m_psr = float(arcpy.management.GetRasterProperties(psr_rast, "MEAN").getOutput(0))
            
            results.append([oid, m_slope, m_north, m_east, m_psr, std_slope])
            processed_count += 1
            
            # --- 实时监控与 Debug 输出 ---
            if processed_count % 10 == 0 or processed_count == total_points:
                elapsed = time.time() - start_time
                speed = processed_count / elapsed # 每秒处理点数
                percent = (processed_count / total_points) * 100
                remaining = total_points - processed_count
                eta_min = (remaining / speed) / 60 if speed > 0 else 0
                
                debug_msg = (f"进度: {percent:.1f}% ({processed_count}/{total_points}) | "
                             f"速度: {speed:.2f} 点/秒 | "
                             f"预计剩余: {eta_min:.1f} 分钟")
                print(debug_msg)
                arcpy.AddMessage(debug_msg)
                
        except Exception as e:
            error_count += 1
            err_log = f"!!! 点 OID {oid} 分析出错: {str(e)}"
            print(err_log)
            arcpy.AddWarning(err_log)

# ==================== 写入 CSV ====================
print(f"正在保存数据至 {output_csv}...")
with open(output_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(results)

total_time = (time.time() - start_time) / 60
print(f"--- 任务结束 ---")
print(f"成功处理: {processed_count} | 失败记录: {error_count}")
print(f"总耗时: {total_time:.2f} 分钟")
arcpy.CheckInExtension("Spatial")